In [6]:
!pip install transformers torch accelerate

In [8]:
import warnings, logging, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)

MODEL_NAME="Qwen/Qwen2.5-1.5B-Instruct"
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 60)
print(f"  Model  : {MODEL_NAME}")
print(f"  Device : {device}")
print("="*40)
print("Loading model and tokenizer...\n")

tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model=AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    device_map="auto"
)
model.eval()
print("Model loaded successfully!\n")

# Response Generation
def generate_response(user_input: str, history: list) -> str:
    messages=[
        {
            "role": "system",
            "content": (
                "You are a helpful and knowledgeable AI assistant chatbot. "
                "Answer questions clearly and accurately in 2-4 sentences. "
                "Provide specific facts, names, and dates when relevant."
            )
        }
    ] + history + [{"role": "user", "content": user_input}]
    text=tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs=tokenizer(text, return_tensors="pt").to(device)
    input_len=inputs["input_ids"].shape[-1]
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens=output_ids[0][input_len:]
    response=tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
    return response if response else "I'm not sure about that. Could you rephrase?"

# Conversation Loop
def run_chatbot():
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("-"*40)
    print("  [Tip] Type 'exit' or 'quit' to end the session.")
    print("-" *40 + "\n")
    history=[]
    while True:
        try:
            user_input=input("You: ").strip()
        except (KeyboardInterrupt, EOFError):
            print("\nChatbot: Session interrupted. Goodbye!")
            break
        if not user_input:
            continue
# Exit Condition
        if user_input.lower() in {"exit", "quit"}:
            print("Chatbot: Thank you for chatting! Goodbye and have a great day!")
            print("\n" + "=" *40)
            print(f"  Session ended. Total turns: {len(history) // 2}")
            print("=" *40)
            break
# Generate and store response
        response=generate_response(user_input, history)
        history.append({"role": "user",      "content": user_input})
        history.append({"role": "assistant", "content": response})
        if len(history) > 20:
            history = history[-20:]
        print(f"Chatbot: {response}\n")
if __name__ == "__main__":
    run_chatbot()

  Model  : Qwen/Qwen2.5-1.5B-Instruct
  Device : cpu
Loading model and tokenizer...



Model loaded successfully!

Chatbot: Hello! I am your AI assistant. How can I help you today?
----------------------------------------
  [Tip] Type 'exit' or 'quit' to end the session.
----------------------------------------

You: Hello
Chatbot: Hello! How can I assist you today?

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence (AI) refers to the simulation of human intelligence processes by computer systems, including learning, reasoning, and self-correction. It encompasses techniques for creating intelligent machines that can perform tasks requiring human-like thinking or decision-making capabilities. AI technologies include machine learning algorithms, neural networks, natural language processing, and expert systems.

You: Who created Python?
Chatbot: Python was created by Guido van Rossum. Van Rossum started working on it while he was at CWI (Centrum Wiskunde & Informatica) in Amsterdam, Netherlands, from 1989 to 1990. He named the language "Python" after Mo